In [ ]:
import os
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root bootstrapped: {repo_root}")


# Mission M41 Lab: Design an Integrated AI System

This notebook guides you through defining system boundaries, decision classifiers, interface SLAs, degradation policies, and telemetry budgets for Flagship V11 (*Evaluated & Observable AI System*).


## Experiment 1: System Boundary & Interface Contract Validation

**Prediction Before Action**:
1. Predict that calling `ArchitectureValidator.validate` on `build_default_v11_architecture()` will return `is_valid == True` with 0 errors.
2. Predict that an empty system boundary list will fail validation with a descriptive error message.


In [ ]:
from missions.M41.integrated_architecture import (
    build_default_v11_architecture,
    ArchitectureValidator,
    SystemArchitectureConfig
)

config = build_default_v11_architecture()
result = ArchitectureValidator.validate(config)
print(f"Default config valid: {result.is_valid}, errors: {result.errors}")

invalid_config = SystemArchitectureConfig(name="Invalid Config", version="1.0")
invalid_result = ArchitectureValidator.validate(invalid_config)
print(f"Invalid config valid: {invalid_result.is_valid}, errors: {invalid_result.errors}")


### MAP (Meaning, Application, Provenance) — Experiment 1
- **Meaning**: System and trust boundaries define exact enforcement layers where non-deterministic AI decisions must be sanitized or verified before side-effect execution.
- **Application**: Guard API gateways and vector stores behind explicit tenant filters and deterministic validators.
- **Provenance**: Derived from Anthropic Agent & Evaluation guidelines and system integration best practices.


## Experiment 2: Telemetry Budget & Degradation Routing

**Prediction Before Action**:
1. Predict that a telemetry trace with latency 450ms and cost $0.008 will pass an ObservabilityBudget configured with 500ms max latency and $0.01 max cost.
2. Predict that a trace violating the latency budget will return `latency_ok == False` during budget evaluation.


In [ ]:
from missions.M41.integrated_architecture import (
    ObservabilityBudget,
    TelemetryTrace,
    DegradationMode
)

budget = ObservabilityBudget(max_latency_ms=500.0, max_cost_usd=0.01)

pass_trace = TelemetryTrace(trace_id="trace-pass-1", latency_ms=450.0, cost_usd=0.008)
print(f"Pass trace budget eval: {pass_trace.evaluate_budget(budget)}")

slow_trace = TelemetryTrace(trace_id="trace-slow-1", latency_ms=750.0, cost_usd=0.008, degradation_mode=DegradationMode.REDUCED_RETRIEVAL)
print(f"Slow trace budget eval: {slow_trace.evaluate_budget(budget)}")


### MAP (Meaning, Application, Provenance) — Experiment 2
- **Meaning**: Observability budgets prevent runaway API costs, latency spikes, and unmonitored failures during production deployment.
- **Application**: Monitor trace budgets continuously and transition to reduced capability modes when SLA thresholds are breached.
- **Provenance**: Integrates V11 evaluation harness and observability governance controls.
